# OttoLatin – B1: Few-shot LLM baseline (eğitimsiz)

Qwen2.5-Instruct modeline 8 örnek gösterip Osmanlıca → Latin çevirisi ister. Eğitim yok.

**Bir kerelik hazırlık (GitHub token):**
1. GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → Generate new token
2. Repository access: *Only select repositories* → `fatiheogan/OttoLatin`; Permissions → **Contents: Read-only** → Generate
3. Colab'da soldaki **🔑 (Secrets)** → *Add new secret* → Name: `GITHUB_TOKEN`, Value: token → *Notebook access* açık

**Çalıştırma:** Çalışma zamanı → T4 GPU → **Tümünü çalıştır**. Sonda `preds_qwen_fewshot.zip` iner.
(Token yoksa 2. hücre `eval.zip` yüklemeni ister.)

- **GPU varsa (T4):** 1.5B model, tüm setler → gerçek B1 sonucu (~30–60 dk)
- **GPU yoksa:** 0.5B model, set başına 20 satır → sadece boru hattı testi, skorlar kullanılmaz


In [ ]:
!pip -q install -U transformers accelerate
import json, os, time, torch, zipfile
from google.colab import files
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU YOK - calisma zamanini T4 yap!")

In [ ]:
# Degerlendirme setlerini al: once GitHub reposundan, olmazsa eval.zip yukleme
EVAL = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not os.path.exists("OttoLatin"):
        !git clone -q https://{token}@github.com/fatiheogan/OttoLatin.git
    if not os.path.isdir("OttoLatin/data/eval"):
        raise RuntimeError("clone basarisiz (token yetkisi?)")
    EVAL = "OttoLatin/data/eval"
    print("Repodan alindi.")
except Exception as e:
    print("Repo alinamadi (", type(e).__name__, ") -> eval.zip yukle")
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("eval_raw")
    # Windows Compress-Archive zip'leri "eval\\x.jsonl" seklinde ters egik cizgi kullanir -> duzelt
    import shutil
    os.makedirs("eval_fixed", exist_ok=True)
    for root, _, fs in os.walk("eval_raw"):
        for f in fs:
            name = f.replace("\\", "/").split("/")[-1]
            if name.endswith(".jsonl"):
                shutil.copy(os.path.join(root, f), os.path.join("eval_fixed", name))
    EVAL = "eval_fixed"
print("eval klasoru:", EVAL, sorted(os.listdir(EVAL)))

In [ ]:
# ---- AYARLAR ----
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cuda":
    MODEL, LIMIT, BATCH, OUT = "Qwen/Qwen2.5-1.5B-Instruct", None, 16, "preds/qwen_fewshot"
else:   # GPU yok -> sadece boru hatti testi
    MODEL, LIMIT, BATCH, OUT = "Qwen/Qwen2.5-0.5B-Instruct", 20, 4, "preds/qwen_cpu_test"
SETS = ["test_2k", "riqa", "naskh", "ot2tr"]
print(f"Cihaz: {DEV} | Model: {MODEL} | LIMIT: {LIMIT}")
os.makedirs(OUT, exist_ok=True)

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32).to(DEV).eval()

load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
shots = load(f"{EVAL}/fewshot.jsonl")
SYSTEM = ("You transliterate Ottoman Turkish written in Arabic script into Modern Turkish Latin script. "
          "Output ONLY the transliteration on one line, nothing else.")

def build(ottoman):
    msgs = [{"role": "system", "content": SYSTEM}]
    for s in shots:
        msgs += [{"role": "user", "content": s["ottoman"]}, {"role": "assistant", "content": s["turkish"]}]
    msgs.append({"role": "user", "content": ottoman})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(build("یارین قبول ایدییورم.")[-400:])

In [ ]:
@torch.no_grad()
def run(rows):
    preds = []
    for i in range(0, len(rows), BATCH):
        b = rows[i:i + BATCH]
        enc = tok([build(r["ottoman"]) for r in b], return_tensors="pt", padding=True).to(DEV)
        max_new = min(512, int(max(len(r["ottoman"]) for r in b) * 1.6) + 16)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        for r, o in zip(b, out[:, enc["input_ids"].shape[1]:]):
            text = tok.decode(o, skip_special_tokens=True).strip().split("\n")[0].strip()
            preds.append({"id": r["id"], "prediction": text})
        if (i // BATCH) % 10 == 0:
            print(f"  {i + len(b)}/{len(rows)}")
    return preds

for s in SETS:
    rows = load(f"{EVAL}/{s}.jsonl")[:LIMIT]
    t = time.time()
    preds = run(rows)
    with open(f"{OUT}/{s}.jsonl", "w", encoding="utf-8") as f:
        for p in preds:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"[{s}] {len(preds)} tahmin, {time.time() - t:.0f} sn")
    for r, p in list(zip(rows, preds))[:3]:
        print("   REF :", r["turkish"][:100]); print("   PRED:", p["prediction"][:100])

In [ ]:
!cd preds && zip -qr ../preds_qwen_fewshot.zip .
files.download("preds_qwen_fewshot.zip")